<a href="https://colab.research.google.com/github/klavarihards/Markets-Strategies/blob/main/portfolio-engine/notebooks/02_allocation_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Engine, Phase 2: Five Allocation Methods under Three Client Profiles

Phase 1 built the data and three baselines. This notebook asks the research question: **which way of choosing portfolio weights holds up best out of sample, for real clients, after costs?**

Five methods, from the simplest to the most elaborate:

| Method | Uses expected returns? | Idea in one line |
|---|---|---|
| Equal weight | No | Same amount in everything. Nothing to estimate, nothing to get wrong. |
| Mean-variance (Ledoit-Wolf) | Historical averages | Best return-for-risk trade-off, *if* the inputs were right. |
| Black-Litterman | Market-implied | Same trade-off, starting from what a neutral market portfolio implies. |
| Risk parity | No | Each sleeve contributes the same amount of risk. |
| Hierarchical risk parity | No | Cluster similar sleeves, then split money down the tree, less to riskier groups. |

Each runs under three client profiles (conservative, balanced, growth), whose limits on equity, bonds and cash, gold, single-sleeve size and volatility every portfolio must respect. Everything is walk-forward with the same costs as Phase 1, and the last two years stay locked.

The reasoning behind each method, and what can go wrong with it, is in `portfolio-engine/HOW_IT_WORKS.md`.

In [ ]:
# Setup: clone the repo (in Colab) and install the package, plus the libraries used for cross-checks.
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("Markets-Strategies"):
    !git clone -q https://github.com/klavarihards/Markets-Strategies.git
if IN_COLAB:
    os.chdir("Markets-Strategies/portfolio-engine")
    !pip install -q -e ".[download,crosscheck]"
    sys.path.insert(0, "src")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portfolio_engine.config import load_config, config_hash
from portfolio_engine.data import load_market_data
from portfolio_engine.estimators import ledoit_wolf, historical_mean
from portfolio_engine.allocation import (RiskProfile, build_method_allocators, risk_contributions,
                                         risk_parity_weights, hrp_weights, ex_ante_vol)
from portfolio_engine.backtest import (run_experiment, summary_table, compliance_table, exposure_table,
                                       sharpe_comparison_table, save_run, metrics, rebalance_dates)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

cfg = load_config("configs/phase2_methods.yaml")
SNAPSHOT = cfg["data"]["snapshot_dir"]        # the same frozen snapshot as Phase 1
print("Config:", cfg["name"], "| hash:", config_hash(cfg))

data = load_market_data(cfg, snapshot_dir=SNAPSHOT)
print(f"Development data: {data.returns.index[0].date()} to {data.returns.index[-1].date()} | "
      f"hold-out locked from {data.holdout_start.date()} | snapshot {data.snapshot_id}")

## Step 1: The client profiles

These are the suitability limits. The ranges are hard limits. The volatility band's upper end is used as a cap on *estimated* volatility whenever it can be met without breaking the hard limits. The maximum loss is what the client says they could live with; it is checked against realised drawdowns below.

In [ ]:
profiles = {p: RiskProfile.from_config(cfg, p) for p in cfg["profiles"]}
pd.DataFrame({p: {**{f"{g} range": f"{lo:.0%}-{hi:.0%}" for g, (lo, hi) in pr.bounds.items()},
                  "vol band": f"{pr.vol_band[0]:.0%}-{pr.vol_band[1]:.0%}",
                  "max loss": f"{pr.max_loss:.0%}", "max single sleeve": f"{pr.max_weight:.0%}"}
              for p, pr in profiles.items()})

## Step 2: One rebalance date, up close

Before the full backtest, look at what each method does on a single date: the last quarter-end of the development period, using the 3-year window. This is exactly the information the backtest would have had on that day.

In [ ]:
w_years = cfg["backtest"]["window_years"][0]
d = rebalance_dates(data.returns.index)[-1]
hist = data.returns.loc[d - pd.DateOffset(years=w_years):d]
cov, delta = ledoit_wolf(hist)
print(f"Date: {d.date()} | {len(hist)} days of history | Ledoit-Wolf shrinkage intensity: {delta:.2f}")

allocs = build_method_allocators(cfg)
weights = pd.DataFrame({name: a(hist) for name, a in allocs.items()}).T
weights.index = pd.MultiIndex.from_tuples([n.split(":") for n in weights.index], names=["profile", "method"])
weights["ex_ante_vol"] = [a.log[-1]["ex_ante_vol"] for a in allocs.values()]
weights.style.format("{:.1%}")

In [ ]:
# Before the profile: what risk parity and HRP choose on their own (cash excluded), and what equal weight is.
risky = [s for s in cov.index if data.asset_class[s] != "cash"]
raw = pd.DataFrame({
    "equal_weight": pd.Series(1 / len(cov), cov.index),
    "risk_parity": pd.Series(risk_parity_weights(cov.loc[risky, risky].to_numpy()), risky),
    "hrp": hrp_weights(cov.loc[risky, risky]),
}).fillna(0.0)
rc = pd.DataFrame({m: risk_contributions(raw[m].to_numpy(), cov.to_numpy()) for m in raw}, index=cov.index)
display(pd.concat({"weight": raw, "share of risk": rc}, axis=1).style.format("{:.1%}"))
print("Risk parity: every risky sleeve carries the same share of risk, so calm assets get large weights.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, p in zip(axes, profiles):
    weights.xs(p, level="profile").drop(columns="ex_ante_vol").plot.bar(ax=ax, stacked=True, legend=(p == "growth"))
    ax.set_title(f"{p}: target weights on {d.date()}")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis="y")
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Step 3: Cross-check against established libraries

Our own optimisers are checked against PyPortfolioOpt and Riskfolio-Lib on this real date (the test suite does the same on synthetic data). The differences should be at the level of solver tolerance.

In [ ]:
import scipy.cluster.hierarchy as sch
if not hasattr(sch, "_LINKAGE_METHODS"):          # PyPortfolioOpt 1.6 needs this name on newer scipy
    sch._LINKAGE_METHODS = {"single": 0, "complete": 1, "average": 2, "weighted": 6}
from pypfopt import EfficientFrontier, HRPOpt
import riskfolio as rf
from portfolio_engine.allocation.optimizer import solve

checks = {}
mu = historical_mean(hist)
for p, pr in profiles.items():
    mine, _ = solve(cov.to_numpy(), list(cov.index), data.asset_class, pr, "utility",
                    mu=mu.to_numpy(), risk_aversion=cfg["methods"]["risk_aversion"], vol_cap=False)
    ef = EfficientFrontier(mu, cov, weight_bounds=(0, pr.max_weight))
    group_of = {s: g for g, cl in pr.groups.items() for s, c in data.asset_class.items() if c in cl}
    ef.add_sector_constraints(group_of, {g: b[0] for g, b in pr.bounds.items()}, {g: b[1] for g, b in pr.bounds.items()})
    ref = pd.Series(ef.max_quadratic_utility(risk_aversion=cfg["methods"]["risk_aversion"])).reindex(cov.index)
    checks[f"mean-variance ({p}) vs PyPortfolioOpt"] = np.abs(mine - ref.to_numpy()).max()

ref_hrp = pd.Series(HRPOpt(cov_matrix=cov.loc[risky, risky]).optimize()).reindex(risky)
checks["HRP vs PyPortfolioOpt"] = np.abs(raw.loc[risky, "hrp"] - ref_hrp).max()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    port = rf.Portfolio(returns=hist[risky]); port.assets_stats(method_mu="hist", method_cov="hist")
    port.cov = cov.loc[risky, risky]
    ref_rp = port.rp_optimization(model="Classic", rm="MV", rf=0, b=None, hist=True)["weights"]
checks["risk parity vs Riskfolio-Lib"] = np.abs(raw.loc[risky, "risk_parity"] - ref_rp.reindex(risky)).max()
pd.Series(checks, name="max weight difference").to_frame().style.format("{:.1e}")

## Step 4: The walk-forward comparison

Every method, under every profile, for both estimation windows, rebalanced quarterly from cash with all costs. This takes about a minute.

In [ ]:
results = run_experiment(cfg, data)
summary = summary_table(results, data, cfg)
pct = ["ann_return", "ann_vol", "max_drawdown", "cvar95_daily", "cvar95_monthly",
       "turnover_per_year", "tracking_error", "cost_drag_per_year"]
fmt = {**{c: "{:.2%}" for c in pct}, "sharpe": "{:.2f}", "sortino": "{:.2f}", "calmar": "{:.2f}"}
summary.style.format(fmt, na_rep="-")

### Did each portfolio behave as the client was promised?

`vol_in_band`: realised volatility inside the profile's band. `max_loss_breached`: the worst drawdown was deeper than the client's stated limit. `share_vol_cap_relaxed`: how often the volatility cap had to be dropped to keep the hard limits. `avg_moved_by_profile`: for equal weight, risk parity and HRP, how much of the portfolio the profile limits had to move away from the method's own choice (0.30 = 30% of the portfolio).

In [ ]:
compliance = compliance_table(results, data, cfg)
compliance.style.format({"realised_vol": "{:.1%}", "vol_band_low": "{:.0%}", "vol_band_high": "{:.0%}", "max_drawdown": "{:.1%}", "max_loss_limit": "{:.0%}", "avg_ex_ante_vol": "{:.1%}", "share_vol_cap_relaxed": "{:.0%}", "avg_moved_by_profile": "{:.0%}"}, na_rep="-")

In [ ]:
exposures = exposure_table(results, data)
exposures.style.format("{:.1%}")

### Are the differences real? Bootstrap intervals for Sharpe differences

For each profile method, the Sharpe ratio minus that of equal weight and of 60/40, with a 90% interval from a block bootstrap (see `metrics.sharpe_difference_ci`). `significant` is True only if the whole interval is on one side of zero. Expect most rows to say False: with this much data, small Sharpe gaps cannot be told apart from luck.

In [ ]:
sharpe_cmp = sharpe_comparison_table(results, data, cfg)
sharpe_cmp.style.format({"sharpe_diff": "{:+.2f}", "ci_low": "{:+.2f}", "ci_high": "{:+.2f}",
                         "share_boot_positive": "{:.0%}"})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
for ax, p in zip(axes, profiles):
    for (win, name), res in results.items():
        prof = name.split(":")[0] if ":" in name else "benchmark"
        if win != w_years or prof not in (p, "benchmark") or name == "taiwan_equity_only":
            continue
        style = "--" if prof == "benchmark" else "-"
        (1 + res.returns).cumprod().plot(ax=ax, label=name.split(":")[-1], linestyle=style)
    ax.set_title(f"{p} ({w_years}-year window), after costs, TWD")
    ax.set_yscale("log"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel("Growth of 1 TWD (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# How the balanced portfolio's weights moved through time under each method.
methods = ["equal_weight", "mean_variance", "black_litterman", "risk_parity", "hrp"]
fig, axes = plt.subplots(len(methods), 1, figsize=(14, 3.2 * len(methods)), sharex=True)
for ax, m in zip(axes, methods):
    results[(w_years, f"balanced:{m}")].weights.plot.area(ax=ax, legend=(m == methods[0]), linewidth=0)
    ax.set_title(f"balanced: {m}"); ax.set_ylim(0, 1); ax.set_ylabel("weight")
axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
tables = {"compliance": compliance, "exposures": exposures, "sharpe_comparison": sharpe_cmp}
run_dir = save_run(results, summary, cfg, data, tables=tables)
print("Saved to", run_dir)

## Step 5: Reproducibility check

Rebuild everything from the frozen snapshot and check that the numbers match exactly (about another minute).

In [ ]:
data2 = load_market_data(load_config("configs/phase2_methods.yaml"), snapshot_dir=SNAPSHOT)
summary2 = summary_table(run_experiment(cfg, data2), data2, cfg)
pd.testing.assert_frame_equal(summary, summary2, check_exact=True)
print("Identical: Phase 2 results reproduce exactly from the snapshot.")

## Findings

*Written after the run on real data. Every number here must come from the tables above.*

- Best and worst method per profile on Sharpe, and whether any gap is significant: [RESULT]
- Did any method beat equal weight and 60/40 reliably on risk (volatility in band, drawdown within the client's limit)? [RESULT]
- How often the profile limits overrode the methods (`avg_moved_by_profile`, `share_vol_cap_relaxed`): [RESULT]
- Turnover and cost drag of mean-variance compared with the others: [RESULT]
- 3-year vs 5-year windows: [RESULT]
- What failed, and why: [RESULT]

## FINAL: hold-out evaluation (do not run during development)

Opens the locked last two years for all methods at once. The plan is to run this **once, at the end of Phase 3**, so leave `RUN_HOLDOUT = False`. Every opening is logged in `results/holdout_log.jsonl`.

In [ ]:
RUN_HOLDOUT = False

if RUN_HOLDOUT:
    from portfolio_engine.backtest import evaluate_holdout
    holdout_summary, holdout_results, holdout_dir = evaluate_holdout(cfg, snapshot_dir=SNAPSHOT)
    print("Saved to", holdout_dir)
    display(holdout_summary)
else:
    print("Hold-out still locked.")